In [1]:
import math 
import numpy as np
import jax.random as jrnd
import jax.numpy as jnp
from jax import grad, jit, jax, lax
from numpy.typing import NDArray
from typing import Callable, NamedTuple, Tuple

In [2]:
def log_p(theta, r):
    return L(theta) - 0.5 * jnp.vdot(r, r)

In [3]:
# this function return 0 if we have a U-Turn
def stop_criterion(theta_minus, theta_plus, r_minus, r_plus) -> jnp.int32:
    delta = theta_plus - theta_minus
    cond = (jnp.dot(delta, r_minus) >= 0) & (jnp.dot(delta, r_plus) >= 0)
    return cond.astype(jnp.int32)

In [4]:
# Leapfrog function
def Leapfrog(theta, r, epsilon):
    # half-step momentum
    _, grad_theta = logprob_and_grad(theta)
    p = r + 0.5 * epsilon * grad_theta

    # full-step position
    q = theta + epsilon * p

    # half-step momentum
    _, grad_q = logprob_and_grad(q)
    p = p + 0.5 * epsilon * grad_q

    return q, p


# Functions to build the trees

In [5]:
class Root(NamedTuple):
    theta   : jnp.ndarray
    r       : jnp.ndarray
    u       : float
    v       : int
    j       : int
    epsilon : float
    theta_0 : jnp.ndarray
    r_0     : jnp.ndarray

class Tree(NamedTuple):
    theta_minus : jnp.ndarray
    r_minus     : jnp.ndarray
    theta_plus  : jnp.ndarray
    r_plus      : jnp.ndarray
    theta_prime : jnp.ndarray
    n_prime     : jnp.ndarray
    s_prime     : jnp.ndarray
    alpha_prime : jnp.ndarray
    n_a_prime   : jnp.ndarray

# Function to build up the base version of the tree
def build_tree_base(root: Root) -> Tree:
    theta, r = root.theta, root.r
    u, v = root.u, root.v
    eps = root.epsilon
    theta0, r0 = root.theta_0, root.r_0

    # leapfrog step
    theta_p, r_p = Leapfrog(theta, r, v * eps)

    logu = jnp.log(u)
    lp = log_p(theta_p, r_p)

    # valid point?
    n_p = (logu <= lp).astype(jnp.int32)

    # non-divergent?
    s_p = (logu < (1000.0 + lp)).astype(jnp.int32)

    # acceptance statistic
    delta = (
        L(theta_p) - 0.5 * jnp.dot(r_p, r_p)
        - L(theta0) + 0.5 * jnp.dot(r0, r0)
    )
    alpha_p = jnp.minimum(1.0, jnp.exp(delta))
    n_a_p = jnp.array(1, dtype=jnp.int32)

    return Tree(
        theta_minus=theta_p,
        r_minus=r_p,
        theta_plus=theta_p,
        r_plus=r_p,
        theta_prime=theta_p,
        n_prime=n_p,
        s_prime=s_p,
        alpha_prime=alpha_p,
        n_a_prime=n_a_p,
    )

In [6]:
# Build the entire tree 

class TreeState(NamedTuple):
    theta_minus: jnp.ndarray
    r_minus: jnp.ndarray
    theta_plus: jnp.ndarray
    r_plus: jnp.ndarray
    theta_prime: jnp.ndarray
    n_prime: jnp.int32
    s_prime: jnp.int32
    alpha_prime: jnp.float32
    n_a_prime: jnp.int32
    key: jnp.ndarray


def BuildTree(root: Root, key: jnp.ndarray):

    def build_base(_):
        return build_tree_base(root), key

    def build_iter(_):
        
        # build the "base level" (depth=0) even if j=0
        base_tree = build_tree_base(root)

        # initialize the state
        state = TreeState(
            theta_minus=base_tree.theta_minus,
            r_minus=base_tree.r_minus,
            theta_plus=base_tree.theta_plus,
            r_plus=base_tree.r_plus,
            theta_prime=base_tree.theta_prime,
            n_prime=base_tree.n_prime,
            s_prime=base_tree.s_prime,
            alpha_prime=base_tree.alpha_prime,
            n_a_prime=base_tree.n_a_prime,
            key=key,
        )

        # this function is going to be call in the for loop for each level j
        def body_fun(depth, state: TreeState):

            # what if the state is valid (s'==1)?
            def expand_valid(state):

                # choose the direction to expand the tree
                theta_start = jnp.where(root.v == -1, state.theta_minus, state.theta_plus)
                r_start     = jnp.where(root.v == -1, state.r_minus,     state.r_plus)

                # set the starting point and expand the tree
                local_root = Root(
                    theta=theta_start,
                    r=r_start,
                    u=root.u,
                    v=root.v,
                    j=0,
                    epsilon=root.epsilon,
                    theta_0=root.theta_0,
                    r_0=root.r_0,
                )

                local_tree = build_tree_base(local_root)

                # update the parameters depending on the choosen direction
                theta_minus = jnp.where(root.v == -1, local_tree.theta_minus, state.theta_minus)
                r_minus     = jnp.where(root.v == -1, local_tree.r_minus,     state.r_minus)
                theta_plus  = jnp.where(root.v ==  1, local_tree.theta_plus,  state.theta_plus)
                r_plus      = jnp.where(root.v ==  1, local_tree.r_plus,      state.r_plus)

                # sample theta_prime from left or right subtree with probability proportional to n_prime
                total_n = state.n_prime + local_tree.n_prime
                p = jnp.where(total_n > 0, local_tree.n_prime / total_n, 0.5)
                key, sub = jrnd.split(state.key)
                choose = jrnd.bernoulli(sub, p)
                theta_prime = jnp.where(choose, local_tree.theta_prime, state.theta_prime)

                # sum the number of valid states of the local tree
                n_prime   = state.n_prime + local_tree.n_prime
                # update the value of s_prime: if stop_criterion is 0 
                # => U-Turn
                # => the tree is not valid anymore
                s_prime   = state.s_prime * local_tree.s_prime * stop_criterion(
                    theta_minus, theta_plus, r_minus, r_plus
                )
                # update the acceptance statistics
                alpha_prime = state.alpha_prime + local_tree.alpha_prime
                n_a_prime   = state.n_a_prime + local_tree.n_a_prime

                # return the state after the aggregation of the new level of the tree
                return TreeState(
                    theta_minus=theta_minus,
                    r_minus=r_minus,
                    theta_plus=theta_plus,
                    r_plus=r_plus,
                    theta_prime=theta_prime,
                    n_prime=n_prime,
                    s_prime=s_prime,
                    alpha_prime=alpha_prime,
                    n_a_prime=n_a_prime,
                    key=key,
                )
                
            # if s'==1 compute expand_valid and return TreeState,
            # otherwise return the state without any changes
            return jax.lax.cond(
                state.s_prime == 1,
                expand_valid,
                lambda s: s,
                operand=state,
            )

        # for loop 
        final_state = jax.lax.fori_loop(1, root.j + 1, body_fun, state)

        # final result of the iteration
        merged = Tree(
            theta_minus=final_state.theta_minus,
            r_minus=final_state.r_minus,
            theta_plus=final_state.theta_plus,
            r_plus=final_state.r_plus,
            theta_prime=final_state.theta_prime,
            n_prime=final_state.n_prime,
            s_prime=final_state.s_prime,
            alpha_prime=final_state.alpha_prime,
            n_a_prime=final_state.n_a_prime,
        )

        return merged, final_state.key

    # choose beetwen base case j=0 and the iterative case
    return jax.lax.cond(
        root.j == 0,
        build_base,
        build_iter,
        operand=None
    )

# Kernel

In [7]:
# Single NUTS step implementation 

class NUTSLoopState(NamedTuple):
    theta_minus: jnp.ndarray
    theta_plus: jnp.ndarray
    r_minus: jnp.ndarray
    r_plus: jnp.ndarray
    theta_prime: jnp.ndarray
    n_prime: jnp.int32
    s_prime: jnp.int32
    alpha_sum: jnp.float32
    n_alpha: jnp.int32
    j: jnp.int32
    key: jnp.ndarray
    u: jnp.float32
    epsilon: jnp.float32
    theta_0: jnp.ndarray
    r_0: jnp.ndarray
    j_max: jnp.int32



# body of the foor loop in the algorithm 3
def NUTS_one_step(theta_0, epsilon, key, j_max = 10):

    # sample momentum
    key, sub_r = jrnd.split(key)
    r_0 = jrnd.normal(sub_r, shape=theta_0.shape)

    # slice variable
    log_joint_0 = log_p(theta_0, r_0)
    key, sub_u = jrnd.split(key)
    u = jnp.exp(log_joint_0) * jrnd.uniform(sub_u)

    # initialize the loop state
    state = NUTSLoopState(
        theta_minus=theta_0,
        theta_plus=theta_0,
        r_minus=r_0,
        r_plus=r_0,
        theta_prime=theta_0,
        n_prime=jnp.array(1, jnp.int32),
        s_prime=jnp.array(1, jnp.int32),
        alpha_sum=jnp.array(0.0),
        n_alpha=jnp.array(0, jnp.int32),
        j=jnp.array(0, jnp.int32),
        key=key,
        u=u,
        epsilon=jnp.asarray(epsilon),
        theta_0=theta_0,
        r_0=r_0,
        j_max=jnp.array(j_max, jnp.int32),
    )


    # condition for the while loop (while (s_prime == 1) & (j < j_max):)
    def cond_fun(state: NUTSLoopState):
        return (state.s_prime == 1) & (state.j < state.j_max)

    # function to execute in the while body
    def body_fun(state: NUTSLoopState):
    
        # Choose a direction
        key, sub_v = jrnd.split(state.key)
        v_j = jnp.where(jrnd.uniform(sub_v) < 0.5, -1, 1)
        
        # Root position/momentum depending on the direction
        root_theta = jnp.where(v_j == -1, state.theta_minus, state.theta_plus)
        root_r = jnp.where(v_j == -1, state.r_minus, state.r_plus)
    
        root = Root(
            theta=root_theta,
            r=root_r,
            u=state.u,
            v=v_j,
            j=state.j,
            epsilon=state.epsilon,
            theta_0=state.theta_0,
            r_0=state.r_0,
        )

        # Call the BuildTree function
        tree, key = BuildTree(root, key)

        # Update bounds
        theta_minus = jnp.where(v_j == -1, tree.theta_minus, state.theta_minus)
        r_minus     = jnp.where(v_j == -1, tree.r_minus,     state.r_minus)
        theta_plus  = jnp.where(v_j ==  1, tree.theta_plus,  state.theta_plus)
        r_plus      = jnp.where(v_j ==  1, tree.r_plus,      state.r_plus)
        
        # Multinomial choice of theta_prime
        key, sub = jrnd.split(key)
        total_n = state.n_prime + tree.n_prime
        p = jnp.where(total_n > 0, tree.n_prime / total_n, 0.5)
        choose = jrnd.bernoulli(sub, p)
        theta_prime = jnp.where(choose, tree.theta_prime, state.theta_prime)

        # update stats
        n_prime   = state.n_prime + tree.n_prime
        s_prime   = state.s_prime * tree.s_prime * stop_criterion(
            theta_minus, theta_plus, r_minus, r_plus
        )
        alpha_sum = state.alpha_sum + tree.alpha_prime
        n_alpha   = state.n_alpha + tree.n_a_prime

        j = state.j + 1

        return NUTSLoopState(
            theta_minus=theta_minus,
            theta_plus=theta_plus,
            r_minus=r_minus,
            r_plus=r_plus,
            theta_prime=theta_prime,
            n_prime=n_prime,
            s_prime=s_prime,
            alpha_sum=alpha_sum,
            n_alpha=n_alpha,
            j=j,
            key=key,
            u=state.u,
            epsilon=state.epsilon,
            theta_0=state.theta_0,
            r_0=state.r_0,
            j_max=state.j_max,
        )

    # perform the while loop and compute the acceptance rate
    final_state = jax.lax.while_loop(cond_fun, body_fun, state)
    accept_rate = final_state.alpha_sum / jnp.maximum(1, final_state.n_alpha)

    #return theta_prime, accept_rate, key
    return final_state.theta_prime, accept_rate, final_state.key


# Sampler without double averaging

In [8]:
def FindReasonableEpsilon(theta: NDArray[float], key=jnp.array([0, 0])) -> float:
    assert theta.ndim == 1

    # sample momentum r ~ N(0, I)
    key, subkey = jrnd.split(key)
    r = jrnd.normal(subkey, shape=theta.shape)

    # initial epsilon
    epsilon = 1.0

    # compute initial log joint density
    E0 = log_p(theta, r)

    # one leapfrog step
    theta_prime, r_prime = Leapfrog(theta, r, epsilon)
    E1 = log_p(theta_prime, r_prime)

    # log acceptance probability
    log_accept = E1 - E0

    # direction: +1 (increase) or -1 (decrease)
    a = jnp.where(log_accept > jnp.log(0.5), 1.0, -1.0)

    # -------------------------------
    # cond_fun: while loop condition
    # -------------------------------
    def cond_fun(state):
        eps, log_acc = state
        return (a * log_acc) > (-a * jnp.log(2.0))

    # --------------------
    # body_fun: loop body
    # --------------------
    def body_fun(state):
        eps, _ = state
        eps = eps * (2.0 ** a)
        theta_p, r_p = Leapfrog(theta, r, eps)
        E1 = log_p(theta_p, r_p)
        log_acc = E1 - E0
        return (eps, log_acc)

    # while loop in JAX
    epsilon, _ = jax.lax.while_loop(cond_fun, body_fun, (epsilon, log_accept))

    return epsilon

In [9]:
def nuts_sampler(theta0, num_samples, key, j_max=10):

    # Set a good value for epsilon
    epsilon = FindReasonableEpsilon(theta0, key)

    # Initialize the chain with zeroes
    samples = jnp.zeros((num_samples, theta0.shape[0]))

    # initialize the initial state
    theta = theta0

    # body function for the for loop
    def body_fun(i, carry):
        theta, key, samples = carry

        # call the kernel function
        theta_new, accept_rate, key = NUTS_one_step(theta, epsilon, key, j_max)

        # save the sampled point
        samples = samples.at[i].set(theta_new)

        return (theta_new, key, samples)

    # for loop 
    theta, key, samples = jax.lax.fori_loop(
        0, num_samples, body_fun, (theta, key, samples)
    )

    return samples, epsilon



In [10]:
# versione jittata
#nuts_sampler_jit = jax.jit(nuts_sampler, static_argnums=(1,))

# Dual averaging

In [11]:
class DualAvgState(NamedTuple):
    log_eps: jnp.float32        # log(epsilon_t)
    log_eps_bar: jnp.float32    # smoothed log epsilon
    H_bar: jnp.float32          # running average of acceptance error
    mu: jnp.float32             # reference point for log epsilon
    t: jnp.int32                # iteration counter
    gamma: jnp.float32          # learning rate
    kappa: jnp.float32          # shrinkage parameter
    tau: jnp.float32            # stabilization parameter

def dual_avg_init(epsilon0: float) -> DualAvgState:
    log_eps0 = jnp.log(epsilon0)

    return DualAvgState(
        log_eps=log_eps0,
        log_eps_bar=log_eps0,
        H_bar=jnp.array(0.0, jnp.float32),
        mu=jnp.log(10.0 * epsilon0),
        t=jnp.array(1, jnp.int32),
        gamma=jnp.array(0.05, jnp.float32),
        kappa=jnp.array(0.75, jnp.float32),
        tau=jnp.array(10.0, jnp.float32),
    )

def dual_avg_update(state: DualAvgState,
                    accept_rate: float,
                    delta: float) -> DualAvgState:

    t = state.t
    gamma = state.gamma
    kappa = state.kappa
    tau = state.tau

    # update H_bar
    H_bar = (1.0 - 1.0 / (t + tau)) * state.H_bar \
            + (1.0 / (t + tau)) * (delta - accept_rate)

    # update log_eps
    log_eps = state.mu - (jnp.sqrt(t) / gamma) * H_bar

    # update smoothed log_eps_bar
    log_eps_bar = (t ** (-kappa)) * log_eps + (1.0 - t ** (-kappa)) * state.log_eps_bar

    # increment t
    t_new = t + 1

    return DualAvgState(
        log_eps=log_eps,
        log_eps_bar=log_eps_bar,
        H_bar=H_bar,
        mu=state.mu,
        t=t_new,
        gamma=gamma,
        kappa=kappa,
        tau=tau,
    )

In [12]:
#warmup
class WarmupConfig(NamedTuple):
    num_warmup: int
    delta: float
    adapt_mass: bool

#config example
#warmup_cfg = WarmupConfig(
#    num_warmup=1000,
#    delta=0.8,
#    adapt_mass=True,
#)

In [13]:
def NUTS_warmup(theta0,
                     num_warmup,
                     key,
                     warmup_cfg: WarmupConfig,
                     j_max=10):

    # 1. Trova epsilon ragionevole
    epsilon0 = FindReasonableEpsilon(theta0, key)

    # 2. Inizializza dual averaging
    da_state = dual_avg_init(epsilon0)

    # 3. Stato iniziale
    theta = theta0

    def body_fun(m, carry):
        theta, key, da_state = carry

        # epsilon_m = exp(log_eps_m)
        epsilon_m = jnp.exp(da_state.log_eps)

        # 1. Un passo NUTS con epsilon_m
        theta_new, accept_rate, key = NUTS_one_step(theta, epsilon_m, key, j_max)

        # 2. Aggiorna dual averaging con alpha_m = accept_rate
        da_state_new = dual_avg_update(da_state, accept_rate, warmup_cfg.delta)

        return (theta_new, key, da_state_new)

    theta, key, da_state = jax.lax.fori_loop(
        0, num_warmup, body_fun, (theta, key, da_state)
    )

    # 4. Step size finale: exp( log_eps_bar_M )
    epsilon_final = jnp.exp(da_state.log_eps_bar)

    return theta, epsilon_final, key


In [14]:
#NUTS_warmup_jit = jax.jit(
#    NUTS_warmup,
#    static_argnums=(1, 3)   # num_warmup, warmup_cfg
#)

In [15]:
def NUTS_sampler_wu(theta0,
                      num_warmup,
                      num_samples,
                      key,
                      warmup_cfg: WarmupConfig,
                      j_max=10):

    # 1. Warmup (Algoritmo 6)
    theta, epsilon_final, key = NUTS_warmup(
        theta0, num_warmup, key, warmup_cfg, j_max
    )

    # 2. Prealloca catena
    samples = jnp.zeros((num_samples, theta0.shape[0]))

    # 3. Sampling con epsilon_final
    def body_fun(i, carry):
        theta, key, samples = carry

        theta_new, accept_rate, key = NUTS_one_step(theta, epsilon_final, key, j_max)

        samples = samples.at[i].set(theta_new)

        return (theta_new, key, samples)

    theta, key, samples = jax.lax.fori_loop(
        0, num_samples, body_fun, (theta, key, samples)
    )

    return samples, epsilon_final

In [16]:
#NUTS_sampler_wu_jit = jax.jit(
#    nuts_sampler_alg6,
#    static_argnums=(1, 2, 4),  # num_warmup, num_samples, warmup_cfg
#)